In [33]:
%use coroutines
@file:Repository("*mavenLocal")
@file:DependsOn("io.github.karloti:concurrent-priority-queue:1.3.4")

In [35]:
import io.github.karloti.cpq.ConcurrentPriorityQueue

val queue = ConcurrentPriorityQueue<Int>(maxSize = 5)

queue.add(10)
queue.add(50)
queue.add(20)
queue.add(5)
queue.add(1)
queue.add(100) // Rejected — larger than all current top 5

queue.items.value  // [1, 5, 10, 20, 50]

[1, 5, 10, 20, 50]

In [36]:
data class SearchResult(val id: String, val score: Int)

val queue = ConcurrentPriorityQueue<SearchResult, String>(
    maxSize = 3,
    comparator = compareByDescending { it.score },  // Higher score = higher priority
    uniqueKeySelector = { it.id }
)

queue.add(SearchResult("A", 10))
queue.add(SearchResult("B", 20))
queue.add(SearchResult("A", 30))  // Updates "A" to score 30 (better priority)
queue.add(SearchResult("A", 5))   // Rejected — existing score 30 is better
queue.add(SearchResult("C", 15))

queue.items.value // [SearchResult(id=A, score=30), SearchResult(id=B, score=20), SearchResult(id=C, score=15)]

[SearchResult(id=A, score=30), SearchResult(id=B, score=20), SearchResult(id=C, score=15)]

In [40]:
data class Task(val id: String, val deadline: Long)

val taskQueue = ConcurrentPriorityQueue<Task, String>(
    maxSize = 100,
    comparator = compareBy { it.deadline },  // Earlier deadline = higher priority
    uniqueKeySelector = { it.id }
)

taskQueue.add(Task("backup", 1735689600))
taskQueue.add(Task("critical-fix", 1735603200))

taskQueue.poll() // Removes and returns Task(id=critical-fix, deadline=1735603200)
taskQueue.items.value // [Task(id=backup, deadline=1735689600)]


[Task(id=backup, deadline=1735689600)]

In [42]:
data class Task(val id: String, val priority: Long)

val queue = ConcurrentPriorityQueue<Task, String>(
    maxSize = 100,
    comparator = compareBy { it.priority },  // Earlier deadline = higher priority
    uniqueKeySelector = { it.id }
)
// Efficiently apply multiple changes without per-operation CAS overhead
val updated = queue.mutate {
    add(Task("task-1", 10))
    add(Task("task-2", 5))
    add(Task("task-3", 20))
    removeIf { it.priority > 15 }
    poll()  // Remove highest priority
}
updated.items.value // `updated` is a new ConcurrentPriorityQueue with all changes applied

[Task(id=task-1, priority=10)]

In [61]:
data class Task(val id: String, val priority: Int, val completed: Boolean)

val queue = ConcurrentPriorityQueue<Task, String>(
    maxSize = 100,
    comparator = compareBy { it.priority },
    uniqueKeySelector = { it.id }
)

// Add multiple elements at once
queue.addAll(
    listOf(
        Task("task-1", 10, false),
        Task("task-2", 5, false),
        Task("task-3", 20, true),
        Task("task-4", 50, false),
    )
)

// All tasks
println("All tasks")
queue.items.value.forEach(::println)

// Remove completed tasks
println("Remove completed tasks")
queue.removeIf { it.completed } // 1
queue.items.value.forEach(::println)

// Keep only high-priority tasks (priority <= 10)
println("Keep only high-priority tasks (priority <= 10)")
queue.retainIf { it.priority <= 10 }
queue.items.value.forEach(::println)

// Poll highest priority task
queue.poll().also(::println)  // Task(id=task-2, priority=5, completed=false)

queue.items.value.also(::println)  // Task(id=task-1, priority=10, completed=false)

// Clear all remaining
queue.clear()

All tasks
Task(id=task-2, priority=5, completed=false)
Task(id=task-1, priority=10, completed=false)
Task(id=task-3, priority=20, completed=true)
Task(id=task-4, priority=50, completed=false)
Remove completed tasks
Task(id=task-2, priority=5, completed=false)
Task(id=task-1, priority=10, completed=false)
Task(id=task-4, priority=50, completed=false)
Keep only high-priority tasks (priority <= 10)
Task(id=task-2, priority=5, completed=false)
Task(id=task-1, priority=10, completed=false)
Task(id=task-2, priority=5, completed=false)
[Task(id=task-1, priority=10, completed=false)]
